<a href="https://colab.research.google.com/github/5ahar-K/CodeGraph-agent/blob/main/Code%20dependencies.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install networkx anthropic

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 12.2 MB/s eta 0:00:00


In [2]:
!git clone https://github.com/pallets/click.git target_repo

Cloning into 'target_repo'...
remote: Enumerating objects: 15423, done.
remote: Counting objects: 100% (531/531), done.
remote: Compressing objects: 100% (226/226), done.
remote: Total 15423 (delta 456), reused 305 (delta 305), pack-reused 14892 (from 4)
Receiving objects: 100% (15423/15423), 5.45 MiB | 19.72 MiB/s, done.
Resolving deltas: 100% (10527/10527), done.


In [3]:
from google.colab import userdata
import google.generativeai as genai

genai.configure(api_key=userdata.get('GEMINI_API_KEY'))
client = genai

/usr/local/lib/python3.12/dist-packages/google/colab/_import_hooks/_hook_injector.py:55: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  loader.exec_module(module)


In [4]:
import ast
import os
import networkx as nx
from collections import defaultdict

def find_function_calls(func_node):
    calls = []
    for node in ast.walk(func_node):
        if isinstance(node, ast.Call):
            if isinstance(node.func, ast.Name):
                calls.append(node.func.id)
            elif isinstance(node.func, ast.Attribute):
                calls.append(node.func.attr)
    return calls

def get_imports(tree):
    imports = {}
    for node in ast.walk(tree):
        if isinstance(node, ast.ImportFrom):
            module = node.module or ""     #from . import echo    (importing from the current package)
            for alias in node.names:     #e.g. from click.utils import echo as e
                local_name = alias.asname or alias.name
                imports[local_name] = module
        elif isinstance(node, ast.Import):
            for alias in node.names:
                local_name = alias.asname or alias.name    #e.g import click (no from package)
                imports[local_name] = alias.name
    return imports

def module_matches_file(module, filepath):
    if not module:
        return False
    module_as_path = module.replace(".", os.sep)
    normalized_filepath = filepath.replace("\\", os.sep)
    return module_as_path in normalized_filepath

def build_graph(repo_path):
    graph = nx.DiGraph()
    name_to_keys = defaultdict(list)
    function_bodies = {}       # (file, name) -> AST node
    file_imports = {}          # file -> {imported_name: module_string}

    # Pass 1: Find all the functions
    for root, _, files in os.walk(repo_path):
        for filename in files:
            if not filename.endswith(".py"):
                continue
            filepath = os.path.join(root, filename)
            try:
                with open(filepath, encoding="utf-8") as f:
                    tree = ast.parse(f.read(), filename=filepath)
            except SyntaxError:
                continue

            file_imports[filepath] = get_imports(tree)

            for item in tree.body:
                if isinstance(item, ast.ClassDef):
                    class_name = item.name
                    for sub_item in item.body:
                        if isinstance(sub_item, ast.FunctionDef):
                            key = (filepath, class_name, sub_item.name)
                            graph.add_node(key, name=sub_item.name, file=filepath,
                                           line=sub_item.lineno, class_name=class_name)
                            name_to_keys[sub_item.name].append(key)
                            function_bodies[key] = sub_item

                elif isinstance(item, ast.FunctionDef):
                    key = (filepath, None, item.name)
                    graph.add_node(key, name=item.name, file=filepath,
                                   line=item.lineno, class_name=None)
                    name_to_keys[item.name].append(key)
                    function_bodies[key] = item

    # Pass 2: map each call to the most likely specific function
    for key, func_node in function_bodies.items():
        source_file, _, _ = key
        imports_in_this_file = file_imports.get(source_file, {})

        for called_name in find_function_calls(func_node):
            candidates = name_to_keys.get(called_name)
            if not candidates:
                continue

            resolved = None

            # 1. Same-file candidates win first (any class, or top-level)
            same_file_candidates = [c for c in candidates if c[0] == source_file]
            if same_file_candidates:
                resolved = same_file_candidates

            # 2. Explicitly imported
            elif called_name in imports_in_this_file:
                module = imports_in_this_file[called_name]
                matched = [c for c in candidates if module_matches_file(module, c[0])]
                if matched:
                    resolved = matched

            # 3. Only one candidate anywhere
            if resolved is None and len(candidates) == 1:
                resolved = candidates

            # 4. Still ambiguous
            if resolved is None:
                resolved = candidates

            for target_key in resolved:
                graph.add_edge(key, target_key)

    return graph, function_bodies, name_to_keys
g, function_bodies, name_to_keys = build_graph("messy_repo1")
ambiguous_count = 0
for key, func_node in function_bodies.items():
    calls = find_function_calls(func_node)
    for name in calls:
        if len(name_to_keys.get(name, [])) > 1:
           ambiguous_count += 1

print(f"Calls to ambiguously-named functions: {ambiguous_count}")

Calls to ambiguously-named functions: 0


In [5]:
ambiguous_calls = 0
resolved_to_one = 0
still_ambiguous = 0

for key, func_node in function_bodies.items():
    for called_name in find_function_calls(func_node):
        candidates = name_to_keys.get(called_name)
        if not candidates or len(candidates) <= 1:
            continue  # not ambiguous to begin with, skip entirely

        ambiguous_calls += 1

        # How many of those candidates actually got an edge in the real graph?
        actual_targets = [c for c in candidates if g.has_edge(key, c)]

        if len(actual_targets) == 1:
            resolved_to_one += 1
        elif len(actual_targets) > 1:
            still_ambiguous += 1

print(f"Calls where multiple same-named functions existed: {ambiguous_calls}")
print(f"  Narrowed down to exactly one target: {resolved_to_one}")
print(f"  Still connected to multiple targets: {still_ambiguous}")

Calls where multiple same-named functions existed: 0
  Narrowed down to exactly one target: 0
  Still connected to multiple targets: 0


In [6]:
def what_calls(graph, filepath, function_name):
    #Who calls this function?
    key = (filepath, function_name)
    return list(graph.predecessors(key))

def what_does_it_call(graph, filepath, function_name):
    #What does this function call?
    key = (filepath, function_name)
    return list(graph.successors(key))

In [7]:
def find_matches(graph, function_name):
    #Return all (file, name) keys in the graph matching this function name.
    return [key for key in graph.nodes if key[1] == function_name]

In [8]:
graph, _, _ = build_graph("target_repo")
matches = find_matches(graph, "echo")
for m in matches:
    print(m)

In [9]:
def get_function_source(graph, key):
    filepath, function_name = key
    with open(filepath, encoding="utf-8") as f:
        source_text = f.read()
    tree = ast.parse(source_text)
    for node in ast.walk(tree):
        if isinstance(node, ast.FunctionDef) and node.name == function_name:
            return ast.get_source_segment(source_text, node)
    return None

def ask_about_function(graph, key, question):
    source = get_function_source(graph, key)
    callers = what_calls(graph, key[0], key[1])
    callees = what_does_it_call(graph, key[0], key[1])

    prompt = f"""Here is a function called `{key[1]}` from file `{key[0]}`:

{source}

It is called by: {callers}
It calls: {callees}

Question: {question}

Answer using only the information given above. If you can't determine the answer from this, say so."""

    model = client.GenerativeModel(model_name="gemini-flash-latest")
    response = model.generate_content(contents=prompt)
    return response.text

def generate_test(graph, key):
    source = get_function_source(graph, key)
    prompt = f"""Write a pytest unit test for this function:

{source}

Only output the test code, no explanation."""
    model = client.GenerativeModel(model_name="gemini-flash-latest")
    response = model.generate_content(contents=prompt)
    return response.text

In [10]:
while True:
    name = input("\nEnter a function name (or 'quit'): ")
    if name == "quit":
        break
    matches = find_matches(graph, name)
    if not matches:
        print("Function not found in graph.")
        continue
    if len(matches) > 1:
        print(f"Multiple functions named '{name}' found:")
        for i, m in enumerate(matches):
            print(f"  [{i}] {m[0]}")
        idx = int(input("Which one? Enter the number: "))
        key = matches[idx]
    else:
        key = matches[0]

    q = input("Your question (or type 'test' to generate a unit test): ")
    if q == "test":
        print(generate_test(g, key))
    else:
        print(ask_about_function(g, key, q))


Enter a function name (or 'quit'): quit


In [15]:
#Messy
!git clone https://github.com/emilybache/GildedRose-Refactoring-Kata messy_repo2
!git clone https://github.com/christianhujer/expensereport messy_repo
!git clone https://github.com/mailpile/Mailpile messy_repo1

fatal: destination path 'messy_repo2' already exists and is not an empty directory.
fatal: destination path 'messy_repo' already exists and is not an empty directory.
fatal: destination path 'messy_repo1' already exists and is not an empty directory.


In [18]:
g2, function_bodies2, name_to_keys2 = build_graph("messy_repo1")
print(f"Found {g2.number_of_nodes()} functions, {g2.number_of_edges()} call relationships")
# for node in g2.nodes(data=True):
#     #print(node)

messy_repo1/shared-data/multipile/mailpile-admin.py:286: SyntaxWarning: invalid escape sequence '\S'
  ps_re = re.compile('^(\S+)\s+(\d+)\s+\S+\s+\S+\s+\S+\s+(\S+)'
messy_repo1/shared-data/multipile/mailpile-admin.py:287: SyntaxWarning: invalid escape sequence '\s'
  '.*\s(?:(?:python[\d\.]*|pypy) +)?'
messy_repo1/shared-data/multipile/mailpile-admin.py:288: SyntaxWarning: invalid escape sequence '\S'
  '(?:\S+/)?(mailpile)(?:\s+|$)')
messy_repo1/shared-data/multipile/mailpile-admin.py:298: SyntaxWarning: invalid escape sequence '\s'
  ns_re = re.compile('^tcp\s+\S+\s+\S+\s+(\S+:\d+)\s+(\S+:.)'
messy_repo1/shared-data/multipile/mailpile-admin.py:299: SyntaxWarning: invalid escape sequence '\s'
  '\s+.*?\s(\d+)\/(\S+)\s*$')
messy_repo1/shared-data/multipile/mailpile-admin.py:595: SyntaxWarning: invalid escape sequence '\.'
  assert(re.match('^[a-z0-9\.]+$', user_settings['host']) is not None)
messy_repo1/shared-data/contrib/forcegrapher/forcegrapher.py:33: SyntaxWarning: invalid escape 

Found 2538 functions, 15078 call relationships


In [19]:
# 1. How many files failed to parse entirely?
parse_failures = []
for root, _, files in os.walk("messy_repo1"):
    for filename in files:
        if not filename.endswith(".py"):
            continue
        filepath = os.path.join(root, filename)
        try:
            with open(filepath, encoding="utf-8") as f:
                ast.parse(f.read(), filename=filepath)
        except SyntaxError as e:
            parse_failures.append((filepath, str(e)))

print(f"Files that failed to parse: {len(parse_failures)}")
for fp, err in parse_failures[:10]:
    print(" -", fp, "|", err)

# 2. Which names are ambiguous, and how ambiguous?
ambiguous_names = {name: keys for name, keys in name_to_keys2.items() if len(keys) > 1}
print(f"\nAmbiguous function names: {len(ambiguous_names)}")
# Sort by how many duplicates exist, worst first
worst = sorted(ambiguous_names.items(), key=lambda kv: -len(kv[1]))
for name, keys in worst[:10]:
    print(f" - '{name}' defined in {len(keys)} places:")
    for k in keys:
        print("     ", k)

# 3. Calls that still fell back to connect to everything (unresolved by the heuristic)
still_ambiguous_calls = []
for key, func_node in function_bodies2.items():
    for called_name in find_function_calls(func_node):
        candidates = name_to_keys2.get(called_name)
        if candidates and len(candidates) > 1:
            actual_targets = [c for c in candidates if g2.has_edge(key, c)]
            if len(actual_targets) > 1:
                still_ambiguous_calls.append((key, called_name, actual_targets))

print(f"\nCalls the heuristic couldn't fully resolve: {len(still_ambiguous_calls)}")
for caller, name, targets in still_ambiguous_calls[:10]:
    print(f" - {caller} calls '{name}', still ambiguous between {len(targets)} candidates")

messy_repo1/shared-data/multipile/mailpile-admin.py:286: SyntaxWarning: invalid escape sequence '\S'
  ps_re = re.compile('^(\S+)\s+(\d+)\s+\S+\s+\S+\s+\S+\s+(\S+)'
messy_repo1/shared-data/multipile/mailpile-admin.py:287: SyntaxWarning: invalid escape sequence '\s'
  '.*\s(?:(?:python[\d\.]*|pypy) +)?'
messy_repo1/shared-data/multipile/mailpile-admin.py:288: SyntaxWarning: invalid escape sequence '\S'
  '(?:\S+/)?(mailpile)(?:\s+|$)')
messy_repo1/shared-data/multipile/mailpile-admin.py:298: SyntaxWarning: invalid escape sequence '\s'
  ns_re = re.compile('^tcp\s+\S+\s+\S+\s+(\S+:\d+)\s+(\S+:.)'
messy_repo1/shared-data/multipile/mailpile-admin.py:299: SyntaxWarning: invalid escape sequence '\s'
  '\s+.*?\s(\d+)\/(\S+)\s*$')
messy_repo1/shared-data/multipile/mailpile-admin.py:595: SyntaxWarning: invalid escape sequence '\.'
  assert(re.match('^[a-z0-9\.]+$', user_settings['host']) is not None)
messy_repo1/shared-data/contrib/forcegrapher/forcegrapher.py:33: SyntaxWarning: invalid escape 

Files that failed to parse: 8
 - messy_repo1/mailpile/commands.py | invalid syntax (commands.py, line 217)
 - messy_repo1/mailpile/util.py | invalid syntax (util.py, line 1153)
 - messy_repo1/mailpile/conn_brokers.py | invalid syntax (conn_brokers.py, line 705)
 - messy_repo1/mailpile/postinglist.py | inconsistent use of tabs and spaces in indentation (postinglist.py, line 131)
 - messy_repo1/mailpile/urlmap.py | invalid syntax (urlmap.py, line 79)
 - messy_repo1/mailpile/plugins/crypto_autocrypt.py | Missing parentheses in call to 'print'. Did you mean print(...)? (crypto_autocrypt.py, line 621)
 - messy_repo1/mailpile/tests/data/pgp-data/buildexamples.py | inconsistent use of tabs and spaces in indentation (buildexamples.py, line 30)
 - messy_repo1/scripts/minimize-pgp-key.py | Missing parentheses in call to 'print'. Did you mean print(...)? (minimize-pgp-key.py, line 14)

Ambiguous function names: 219
 - '__init__' defined in 137 places:
      ('messy_repo1/shared-data/mailpile-gui/

In [20]:
import ast

with open("messy_repo2/python/gilded_rose.py") as f:
    tree = ast.parse(f.read())

for node in ast.walk(tree):
    if isinstance(node, ast.ClassDef):
        print(f"Class: {node.name}")
        for item in node.body:
            if isinstance(item, ast.FunctionDef):
                print(f"  - {item.name} at line {item.lineno}")


Class: GildedRose
  - __init__ at line 5
  - update_quality at line 8
Class: Item
  - __init__ at line 40
  - __repr__ at line 45
